# §1.1.3 — 경험적 위험은 얼마나 빨리 참 위험에 수렴하는가

> 딥러닝 교재 · 1부 1장 1절 4항 (🐍)
> 선행: §1.1.1(정의·i.i.d.) · §1.1.2(불편성과 선택 편향) · §1.1.3(호에프딩 + 합집합 경계)
> 부록 E(확률 부등식) · 부록 K(노트북 사용 안내)

## 이 노트북이 답하는 질문

1. **R(f) 자체를 우리는 어떻게 아는가?** 참값을 아는 합성 자료에서 폐형식과 몬테카를로 근사를 대조한다.
2. **|R̂ₙ(f) − R(f)|는 정말 n^(−1/2)로 줄어드는가?** 로그–로그 축에서 기울기가 −1/2인지, 그리고 **상수까지** 이론과 맞는지 본다.
3. **sup_{f∈H}|R̂ₙ − R|에서 log M은 어디로 들어가는가?** 그리고 §1.1.3 ⚠︎(b)의 주장 — 합집합 경계가 후보 간 상관을 낭비한다 — 이 실제로 관측되는가?
4. **§1.1.2의 낙관 편향이 예측한 크기로 나타나는가?** 참 오차율 0.5짜리 후보 1000개에서 훈련 오차 0.34가 정말 나오는가?

**예상 실행 시간** CPU 단일 코어 약 60초 (`FAST = True`이면 약 15초).
이 실험을 이해하려면 §1.1.2의 정리 2와 §1.1.3의 조립 논증이 필요합니다. 코드가 유도를 대체하지 않습니다.

---
## 0. 설정

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, binom

_t_start = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────────
FAST  = False      # True면 시행 수를 줄여 빠르게
SEED  = 20260730
MU    = 0.75       # 두 클래스 평균이 ±MU. 분리도 d = 2*MU
# ────────────────────────────────────────────────────────────

# 색맹 안전 팔레트 (Okabe–Ito) — 규약 §II.4-7
CB = ['#000000', '#E69F00', '#56B4E9', '#009E73',
      '#D55E00', '#0072B2', '#CC79A7', '#F0E442']
plt.rcParams.update({
    'figure.dpi': 120, 'font.size': 10, 'axes.grid': True,
    'grid.alpha': 0.3, 'axes.prop_cycle': plt.cycler(color=CB),
    'figure.autolayout': True,
})

# 한글 폰트 (부록 K). 없으면 그림 라벨만 영문으로 대체한다 — 어느 환경에서도 읽히게.
import matplotlib.font_manager as fm
_avail = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic', 'Malgun Gothic', 'AppleGothic',
                            'Noto Sans CJK KR', 'Noto Sans KR', 'NanumBarunGothic']
                if f in _avail), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False   # 한글 폰트의 마이너스 글리프 문제
def lab(ko, en):
    return ko if KO_FONT else en

print(f"numpy {np.__version__}  |  FAST={FAST}  |  seed={SEED}")
print(f"한글 폰트: {KO_FONT or '없음 → 그림 라벨은 영문으로 출력됩니다'}")

---
## 1. 참값을 아는 합성 자료

**자료 생성 분포 p.** §1.1.1의 정의를 그대로 구현한다.

$$y \sim \mathrm{Bernoulli}(1/2), \qquad x \mid y \sim \mathcal{N}\big((2y-1)\mu,\ 1\big)$$

**가설 공간 H.** 임계값 분류기 $f_\theta(x) = \mathbb{1}\{x > \theta\}$.

**손실.** 0-1 손실. 따라서 §1.1.2의 주석대로 위험 = 오분류 확률이다.

**참 위험은 폐형식으로 나온다** — 이것이 합성 자료를 쓰는 이유다(규약 §II.4-5).

$$R(\theta) = \tfrac12\,\mathbb{P}(x>\theta \mid y=0) + \tfrac12\,\mathbb{P}(x\le\theta \mid y=1)
           = \tfrac12\big[1-\Phi(\theta+\mu)\big] + \tfrac12\,\Phi(\theta-\mu)$$

$\theta^\*=0$에서 최소가 되고 $R^\* = \Phi(-\mu) = \Phi(-d/2)$ — **§1.2.3에서 손으로 계산할 바로 그 값**이다.

In [ ]:
def risk_exact(theta, mu=MU):
    # 참 위험 R(theta). 폐형식.
    theta = np.asarray(theta, dtype=float)
    return 0.5 * (1.0 - norm.cdf(theta + mu)) + 0.5 * norm.cdf(theta - mu)

def sample(n, rng, mu=MU):
    # 자료 생성 분포 p에서 n개를 i.i.d.로 뽑는다 (= p^{⊗n} 에서 한 번 뽑는다)
    y = rng.integers(0, 2, size=n)
    x = rng.normal(loc=np.where(y == 1, mu, -mu), scale=1.0)
    return x, y

R_star = risk_exact(0.0)
print(f"베이즈 위험  R* = R(0)      = {R_star:.6f}")
print(f"확인:        Phi(-mu)      = {norm.cdf(-MU):.6f}   (§1.2.3의 Phi(-d/2))")

THETA0 = 0.5                       # 이하 '고정된 f'로 쓸 후보 (일부러 최적이 아님)
R0 = float(risk_exact(THETA0))
print(f"고정 후보    R(theta=0.5)   = {R0:.6f}")

### 1.1 경험적 위험 — 임계값 격자 전체를 한 번에

임계값 족에서는 정렬 + 이분 탐색으로 격자 M개의 R̂ₙ을 $O((n+M)\log n)$에 모두 얻는다.
무식하게 계산한 것과 일치하는지 **먼저 확인**한다(규약 §II.4-2의 (a) 단계).

In [ ]:
def emp_risk_thresholds(x, y, thetas):
    # R_hat_n(theta) 를 격자 thetas 전체에 대해 계산
    n = len(x)
    x0 = np.sort(x[y == 0])
    x1 = np.sort(x[y == 1])
    err0 = len(x0) - np.searchsorted(x0, thetas, side='right')   # x>theta 인데 y=0
    err1 = np.searchsorted(x1, thetas, side='right')             # x<=theta 인데 y=1
    return (err0 + err1) / n

_rng = np.random.default_rng(SEED)
_x, _y = sample(500, _rng)
_th = np.linspace(-3, 3, 37)
_brute = np.array([np.mean((_x > t).astype(int) != _y) for t in _th])
assert np.allclose(_brute, emp_risk_thresholds(_x, _y, _th)), "빠른 구현이 무식한 구현과 다름"
print("정합성 확인 통과: 빠른 구현 == 무식한 구현")

---
## 2. R을 몬테카를로로 근사하기 — **참 위험도 추정량이다**

실전에서 p는 모르므로 R은 폐형식으로 나오지 않는다. 대신 **거대한 독립 시험 표본**으로 근사한다.

$$R(f) \approx \frac{1}{N}\sum_{k=1}^{N} \ell\big(f(\tilde x_k), \tilde y_k\big), \qquad N \gg n$$

여기서 배울 것: **이 근사 자체가 R̂과 같은 종류의 추정량**이며 같은 $N^{-1/2}$ 법칙을 따른다.
"R을 안다"는 말은 언제나 "충분히 좁은 오차 막대 안에서 안다"는 뜻이다.

In [ ]:
Ns = np.array([10**3, 10**4, 10**5, 10**6, 10**7])
rng = np.random.default_rng(101)
mc_err = []
for N in Ns:
    xs, ys = sample(int(N), rng)
    R_mc = np.mean((xs > THETA0).astype(int) != ys)
    mc_err.append(abs(R_mc - R0))
    print(f"N={N:>9,}   R_MC={R_mc:.6f}   |R_MC - R|={mc_err[-1]:.6f}")
mc_err = np.array(mc_err)

sigma0 = np.sqrt(R0 * (1 - R0))                 # 0-1 손실의 표준편차
fig, ax = plt.subplots(figsize=(5.2, 3.6))
ax.loglog(Ns, mc_err, 'o-', color=CB[5], label=lab('실측 $|R_{MC}-R|$', 'observed $|R_{MC}-R|$'))
ax.loglog(Ns, sigma0*np.sqrt(2/(np.pi*Ns)), '--', color=CB[0],
          label=lab(r'이론 $\sigma\sqrt{2/(\pi N)}$', r'theory $\sigma\sqrt{2/(\pi N)}$'))
ax.set_xlabel(lab('시험 표본 수 $N$ (개)', 'test sample size $N$'))
ax.set_ylabel(lab('절대 오차 (오분류율 단위)', 'absolute error (0-1 loss units)'))
ax.set_title(lab('몬테카를로로 근사한 $R$도 $N^{-1/2}$로만 정확해진다',
                 'the Monte-Carlo estimate of $R$ also converges only as $N^{-1/2}$'), fontsize=10)
ax.legend(); plt.show()

> **읽는 법.** 점 하나하나가 시행 한 번이라 위아래로 튄다. 추세만 볼 것.
> $N=10^7$에서도 오차는 $10^{-4}$ 수준이며 **0이 아니다.**
> 아래 3절에서는 폐형식 R을 쓰므로 이 오차가 결과를 오염시키지 않는다.

---
## 3. 고정된 f 하나 — 기울기 −1/2와 **상수까지** 맞추기

§1.1.2 정리 1의 조건(f가 표본과 무관)을 지킨 상태다. 0-1 손실이므로 $n\hat R_n(f)\sim\mathrm{Bin}(n, R(f))$이고,
정규 근사에서 평균절대편차는 $\mathbb{E}|Z|=\sqrt{2/\pi}$를 써서

$$\mathbb{E}\big|\hat R_n(f) - R(f)\big| \;\approx\; \sigma(f)\sqrt{\frac{2}{\pi n}},
\qquad \sigma(f)=\sqrt{R(f)\big(1-R(f)\big)}$$

**기울기만 맞추는 것으로 만족하지 말 것.** 상수까지 맞아야 우리가 옳은 것을 재고 있다는 증거가 된다.

In [ ]:
T2 = 150 if FAST else 500
ns = np.unique(np.round(np.logspace(1.3, 3.7, 12)).astype(int))

rng = np.random.default_rng(SEED)
mad = np.empty(len(ns))
for i, n in enumerate(ns):
    xs = rng.normal(size=(T2, n))
    ys = rng.integers(0, 2, size=(T2, n))
    xs = np.where(ys == 1, xs + MU, xs - MU)
    Rhat = np.mean((xs > THETA0).astype(int) != ys, axis=1)
    mad[i] = np.mean(np.abs(Rhat - R0))

slope, intercept = np.polyfit(np.log(ns), np.log(mad), 1)
theory = sigma0 * np.sqrt(2/(np.pi*ns))
print(f"적합된 기울기 = {slope:+.4f}   (이론 -0.5)")
print(f"실측/이론 비  = {np.round(mad/theory, 3)}")

fig, ax = plt.subplots(figsize=(5.2, 3.8))
ax.loglog(ns, mad, 'o', color=CB[5], label=lab(f'실측 (시행 {T2}회 평균)', f'observed (mean of {T2} runs)'))
ax.loglog(ns, theory, '-', color=CB[0], label=lab(r'이론 $\sigma\sqrt{2/(\pi n)}$', r'theory $\sigma\sqrt{2/(\pi n)}$'))
ax.loglog(ns, np.exp(intercept)*ns**slope, ':', color=CB[4],
          label=lab(f'적합 기울기 {slope:+.3f}', f'fitted slope {slope:+.3f}'))
ax.set_xlabel(lab('훈련 표본 수 $n$ (개)', 'training sample size $n$'))
ax.set_ylabel(r'$\mathbb{E}\,|\hat R_n(f)-R(f)|$')
ax.set_title(lab('고정된 $f$: 기울기 $-1/2$가 상수까지 맞는다',
                 'fixed $f$: slope $-1/2$, and the constant matches too'), fontsize=10)
ax.legend(fontsize=8); plt.show()

---
## 4. sup over H — log M은 **절편**으로 들어간다

§1.1.3의 경계는

$$\sup_{f\in H}\big|\hat R_n(f)-R(f)\big| \;\le\; \sqrt{\frac{\log(2M/\delta)}{2n}}$$

로그–로그 축에서 이 식은 **기울기 −1/2의 직선이고 M은 절편만 바꾼다.** 예측이 둘이다.

- (i) M을 키워도 **기울기는 변하지 않는다**
- (ii) 곡선은 위로 **평행 이동**하며, 이동량은 $\sqrt{\log M}$에 비례한다

(i)은 맞을 것이다. **(ii)가 이 절의 진짜 시험대**다 — §1.1.3 ⚠︎(b)는 합집합 경계가 후보 간 상관을
전혀 쓰지 않으므로 상관이 큰 H에서는 과하게 비관적일 것이라고 예고했다. 두 가설족을 나란히 놓고 확인한다.

| 가설족 | 구성 | 후보 간 상관 |
|---|---|---|
| **A. 임계값** | $f_\theta(x)=\mathbb{1}\{x>\theta\}$, θ를 격자로 M개 | **큼** — 이웃한 θ는 거의 같은 함수 |
| **B. 무작위 특징** | $f_j(x)=\mathbb{1}\{\sin(\omega_j x + b_j)>0\}$, ω를 크게 | **거의 없음** — 참 위험은 모두 ≈ 0.5 |

B는 폐형식이 없으므로 **2절의 몬테카를로로 참 위험을 구한다.** 두 방식이 한 노트북에서 함께 쓰이는 지점이다.

In [ ]:
# --- 가설족 B: 무작위 특징. 참 위험은 몬테카를로로 ---
def make_random_features(M, rng):
    w = rng.uniform(50, 150, size=M) * rng.choice([-1, 1], size=M)
    b = rng.uniform(0, 2*np.pi, size=M)
    return w, b

def predict_rf(x, w, b):
    return (np.sin(np.outer(x, w) + b) > 0).astype(np.int8)     # (n, M)

M_MAX = 1000
rng = np.random.default_rng(7)
W, B = make_random_features(M_MAX, rng)

N_MC = 100_000 if FAST else 200_000
xs, ys = sample(N_MC, np.random.default_rng(11))
R_true_rf = np.mean(predict_rf(xs, W, B) != ys[:, None], axis=0)
del xs, ys
print(f"B족 참 위험 (MC, N={N_MC:,}): 평균 {R_true_rf.mean():.4f}, "
      f"범위 [{R_true_rf.min():.4f}, {R_true_rf.max():.4f}]")
print("→ 후보들의 실력 차가 거의 없다. §1.1.2에서 편향이 최대가 되는 조건.")

In [ ]:
Ms  = [2, 10, 100, 1000]
ns3 = np.unique(np.round(np.logspace(1.7, 3.4, 8)).astype(int))
T3  = 40 if FAST else 120

def sup_gap_A(n, M, T, rng):
    thetas = np.linspace(-3, 3, M); Rt = risk_exact(thetas)
    return np.mean([np.max(np.abs(emp_risk_thresholds(*sample(n, rng), thetas) - Rt))
                    for _ in range(T)])

def sup_gap_B(n, M, T, rng):
    out = np.empty(T)
    for t in range(T):
        x, y = sample(n, rng)
        Rh = np.mean(predict_rf(x, W[:M], B[:M]) != y[:, None], axis=0)
        out[t] = np.max(np.abs(Rh - R_true_rf[:M]))
    return out.mean()

res = {'A': {}, 'B': {}}
for M in Ms:
    res['A'][M] = np.array([sup_gap_A(n, M, T3, np.random.default_rng(SEED+M)) for n in ns3])
    res['B'][M] = np.array([sup_gap_B(n, M, max(20, T3//2), np.random.default_rng(SEED+M)) for n in ns3])
    sA, _ = np.polyfit(np.log(ns3), np.log(res['A'][M]), 1)
    sB, _ = np.polyfit(np.log(ns3), np.log(res['B'][M]), 1)
    print(f"M={M:>5}   A(임계값) 기울기 {sA:+.3f}   B(무작위) 기울기 {sB:+.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.4, 3.9), sharey=True)
titles = [lab('A. 임계값 족 — 후보끼리 닮았다', 'A. thresholds - candidates are similar'),
          lab('B. 무작위 특징 족 — 후보끼리 무관하다', 'B. random features - nearly independent')]
for ax, key, ttl in zip(axes, ['A', 'B'], titles):
    for j, M in enumerate(Ms):
        ax.loglog(ns3, res[key][M], 'o-', ms=3.5, color=CB[j+1], label=f'M = {M}')
    ax.loglog(ns3, np.sqrt(np.log(2*1000/0.05)/(2*ns3)), 'k--', lw=1,
              label=lab(r'§1.1.3 경계 (M=1000, $\delta$=.05)', r'§1.1.3 bound (M=1000, $\delta$=.05)'))
    ax.set_xlabel(lab('훈련 표본 수 $n$ (개)', 'training sample size $n$'))
    ax.set_title(ttl, fontsize=10)
axes[0].set_ylabel(r'$\mathbb{E}\,\sup_{f\in H}|\hat R_n(f)-R(f)|$')
axes[1].legend(fontsize=7.5)
fig.suptitle(lab('log M은 기울기가 아니라 절편에 들어간다 — 그 양은 H의 상관에 달렸다',
                 'log M enters the intercept, not the slope - by an amount set by correlation in H'),
             y=1.04, fontsize=10)
plt.show()

n_ref = ns3[-1]
print(f"\nn={n_ref}에서 M=2 → M=1000일 때 sup의 증가 배율")
print(f"  합집합 경계의 예측  sqrt(log1000/log2) = {np.sqrt(np.log(1000)/np.log(2)):.2f} 배")
print(f"  A 임계값 족 (상관 큼)               = {res['A'][1000][-1]/res['A'][2][-1]:.2f} 배")
print(f"  B 무작위 족 (거의 독립)             = {res['B'][1000][-1]/res['B'][2][-1]:.2f} 배")

> ### 이 그림이 말하는 것 — §1.1.3 ⚠︎(b)의 실험적 확인
>
> 두 패널 모두 **기울기는 −1/2로 같다.** 예측 (i)은 맞았다.
>
> 그러나 M을 2에서 1000으로 500배 늘렸을 때 sup이 커지는 정도는 두 족이 다르다.
> 합집합 경계가 예측하는 $\sqrt{\log 1000/\log 2}\approx 3.16$배에 **B는 근접하고 A는 절반 수준**에 그친다.
> 임계값 족에서는 이웃한 후보들이 거의 같은 함수여서 **유효 후보 수가 M보다 훨씬 작기** 때문이다.
>
> **합집합 경계가 느슨해지는 것은 상수 낭비가 아니라 구조를 안 보기 때문이다.**
> 이 낭비를 고치려는 것이 라데마허 복잡도·커버링 수이며, 33장의 절반이 그 이야기다.
> 동시에 검은 점선(경계)이 두 실측선보다 **한참 위에 있다**는 것도 함께 볼 것 — 경계는 어디까지나 상한이다.

---
## 5. §1.1.2 회수 — 승자의 저주를 눈으로

가설족 B는 참 위험이 모두 ≈ 0.5다. 즉 **정보가 전혀 없는 후보 1000개**다.
그중 훈련 오차 최소인 것을 고르면 무슨 일이 벌어지는가.

§1.1.2의 예측: $\mathbb{E}[\hat R_n(\hat f_n)] \approx r - \sigma\,\mathbb{E}[\max_j Z_j]/\sqrt{n}$, 참 위험은 그대로 $r$.
CLT를 쓰지 않는 **정확한 이항 계산**과도 대조한다.

$$\mathbb{E}\Big[\min_j \tfrac1n B_j\Big] = \frac1n\sum_{k=0}^{n-1}\big(1-F_{\mathrm{Bin}(n,1/2)}(k)\big)^M$$

In [ ]:
n4, M4 = 100, 1000
T4 = 800 if FAST else 2500

rng = np.random.default_rng(4242)
train_win = np.empty(T4); true_win = np.empty(T4)
for t in range(T4):
    x, y = sample(n4, rng)
    Rh = np.mean(predict_rf(x, W[:M4], B[:M4]) != y[:, None], axis=0)
    j = int(np.argmin(Rh))
    train_win[t], true_win[t] = Rh[j], R_true_rf[j]

k = np.arange(n4)
exact_min = float(np.sum((1 - binom.cdf(k, n4, 0.5))**M4) / n4)
Zmax = np.max(np.random.default_rng(0).normal(size=(20000, M4)), axis=1).mean()

print(f"실측  E[승자의 훈련 오차] = {train_win.mean():.4f}   → 정확도 {1-train_win.mean():.1%}")
print(f"실측  E[승자의 참 위험]   = {true_win.mean():.4f}   → 정확도 {1-true_win.mean():.1%}")
print(f"      (후보 전체의 평균 참 위험 {R_true_rf[:M4].mean():.4f})")
print( "-"*58)
print(f"정확한 이항 계산 E[min]          = {exact_min:.4f}")
print(f"CLT + E[max Z]={Zmax:.3f} 근사    = {0.5 - 0.5*Zmax/np.sqrt(n4):.4f}")
print(f"CLT + sqrt(2 log M) 상한 근사    = {0.5 - 0.5*np.sqrt(2*np.log(M4)/n4):.4f}  ← 과대 추정")

In [ ]:
fig, ax = plt.subplots(figsize=(5.6, 3.8))
ax.hist(train_win, bins=np.arange(0.25, 0.475, 0.01), color=CB[2],
        edgecolor='w', label=lab('승자의 훈련 오차', 'winner: training error'))
ax.axvline(train_win.mean(), color=CB[5], lw=2,
           label=lab(f'평균 {train_win.mean():.3f}', f'mean {train_win.mean():.3f}'))
ax.axvline(0.5, color=CB[4], lw=2.5,
           label=lab('참 위험 0.500 (변하지 않음)', 'true risk 0.500 (unchanged)'))
ax.axvline(exact_min, color=CB[0], ls=':', lw=1.8,
           label=lab(f'이항 계산 예측 {exact_min:.3f}', f'exact binomial {exact_min:.3f}'))
ax.set_xlim(0.25, 0.55)
ax.set_xlabel(lab('오분류율', '0-1 loss'))
ax.set_ylabel(lab('시행 횟수 (회)', 'count'))
ax.set_title(lab(f'후보 M={M4}개, n={n4}: 실력 0인 승자가 정확도 {1-train_win.mean():.0%}로 보인다',
                 f'M={M4}, n={n4}: a zero-skill winner looks {1-train_win.mean():.0%} accurate'), fontsize=10)
ax.legend(fontsize=8); plt.show()

> ### 이 그림이 이 노트북의 결론이다
>
> 빨간 선(참 위험 0.5)과 파란 분포(훈련 오차 ≈ 0.34) 사이의 간격 **0.16에는 실력이 한 방울도 없다.**
> 후보 1000개를 뒤졌다는 사실만으로 생긴 간격이다.
>
> 그리고 이 간격은 §1.1.2의 폐형식이 예측한 것과 소수 셋째 자리까지 맞는다.
> $\sqrt{2\log M}$을 쓴 값(0.314)이 실측(0.343)보다 낮게 나오는 것도 예상대로다 — 그것은 $\mathbb{E}[\max_j Z_j]$의 **상한**이기 때문이다.
> 교재 본문에는 실측·이항 계산과 일치하는 **0.34**를 쓰고, $\sqrt{2\log M}$은 상한임을 명시할 것.

---
## 6. 3단 원칙 (b) — PyTorch 재작성과 수치 일치 확인

규약 §II.4-2: NumPy로 원리를 구현한 뒤 PyTorch로 다시 쓰고 **같은 수가 나오는지 확인**한다.
여기서는 새로 배울 것이 없으므로 목적은 하나다 — 프레임워크가 바뀌어도 정의는 바뀌지 않는다는 확인.
torch가 없으면 이 셀은 건너뛴다.

In [ ]:
try:
    import torch
    torch.manual_seed(SEED)

    def emp_risk_torch(x, y, thetas):
        pred = (x[:, None] > thetas[None, :]).to(torch.int8)     # (n, M)
        return (pred != y[:, None]).to(torch.float64).mean(dim=0)

    rng = np.random.default_rng(2024)
    x_np, y_np = sample(2000, rng)
    th_np = np.linspace(-3, 3, 257)

    r_np = emp_risk_thresholds(x_np, y_np, th_np)
    r_pt = emp_risk_torch(torch.from_numpy(x_np),
                          torch.from_numpy(y_np),
                          torch.from_numpy(th_np)).numpy()

    max_dev = np.abs(r_np - r_pt).max()
    assert max_dev < 1e-12, max_dev
    print(f"torch {torch.__version__}: 최대 편차 {max_dev:.2e}  → 수치 일치 확인")
except ModuleNotFoundError:
    print("torch 미설치 — 건너뜀. (부록 K의 requirements.txt 참조)")

---
## 7. 자기 점검

답을 적어 본 뒤 셀을 실행해 확인하십시오.

1. 3절의 로그–로그 직선에서 **절편**은 무엇이 정하는가? MU를 0.75에서 1.5로 바꾸면 직선은 어느 방향으로 움직이겠는가?
2. 4절 A족에서 M을 늘려도 sup이 어느 값 이상 커지지 않는 이유는? (힌트: 임계값 격자를 무한히 촘촘하게 하면 H는 무엇이 되는가)
3. 5절에서 **n을 400으로 늘리면** 승자의 훈련 오차는 얼마가 되겠는가? 예측한 뒤 손잡이를 바꿔 확인하라.
4. 5절 실험의 무작위성은 어디서 오는가 — W·B에서인가, 표본에서인가? 둘 중 하나를 고정하면 결과가 달라지는가?

In [ ]:
# 자기 점검 3의 확인 — n만 바꿔 승자의 훈련 오차를 다시 잰다
for n_try in [100, 400, 1600]:
    rng = np.random.default_rng(555)
    v = []
    for _ in range(200 if FAST else 500):
        x, y = sample(n_try, rng)
        Rh = np.mean(predict_rf(x, W[:M4], B[:M4]) != y[:, None], axis=0)
        v.append(Rh.min())
    kk = np.arange(n_try)
    ex = float(np.sum((1 - binom.cdf(kk, n_try, 0.5))**M4) / n_try)
    print(f"n={n_try:>5}   실측 {np.mean(v):.4f}   이항 계산 {ex:.4f}   "
          f"참 위험과의 간격 {0.5-np.mean(v):.4f}")
print("\n→ 간격이 n^(-1/2)로 줄어든다: 400은 100의 절반, 1600은 100의 1/4 근처")

---
## 8. 직접 바꿔 볼 손잡이

셀 상단의 값을 바꾸고 전체 재실행(Kernel → Restart & Run All)하십시오.
**셀 실행 순서에 의존하지 않도록** 작성되어 있습니다(규약 §II.4-4).

| 손잡이 | 위치 | 기본값 | 바꾸면 무슨 일이 |
|---|---|---|---|
| `MU` | 0절 | 0.75 | 두 클래스의 분리도. 크면 $R^\*$가 0에 가까워지고 σ가 줄어 모든 곡선이 아래로 |
| `THETA0` | 1절 | 0.5 | 고정 후보의 위치. $R(\theta_0)$가 0.5에 가까울수록 σ가 커져 3절 곡선이 위로 |
| `Ms` | 4절 | [2,10,100,1000] | 후보 수. 10000까지 늘려 A족의 포화를 확인할 것 |
| `n4`, `M4` | 5절 | 100, 1000 | 승자의 저주 크기. $M4=1$로 두면 편향이 사라지는 것을 확인 |
| `SEED` | 0절 | 20260730 | 시드. 결론이 시드에 의존하지 않아야 한다 |
| `FAST` | 0절 | False | True면 시행 수를 줄여 15초에 완주 |

**권하는 첫 실험** — `M4 = 1`로 두고 5절만 다시 실행하십시오. 히스토그램이 0.5를 중심으로 모입니다.
편향을 만드는 것이 **모형의 복잡도가 아니라 선택 행위**라는 §1.1.2의 문장이 한 줄 수정으로 확인됩니다.

In [ ]:
print(f"총 실행 시간: {time.time() - _t_start:.1f}초")